# OpenRCA — Seconda domanda di ricerca

## Audit preliminare delle 15 query in `ood_holdout`

Questo notebook non interroga i modelli, non legge ground truth e non fa inferenze. Mostra conteggi e sovrapposizioni temporali **prima** di definire il nuovo protocollo discovery/test. Eseguire dopo i test offline in PyCharm. Il clone del repository viene creato in una cartella locale separata.

**Non interpretare 15 query come 15 guasti indipendenti o come 15 regole verificabili.**


In [ ]:
# FASE 2: inventory OOD GT-free. Una sola cella, ZERO chiamate LLM.
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/alessio1996A/Tesi_Laurea_Magistrale_OpenRCA.git"
BRANCH = "feature/ood-knowledge-transfer-audit-2026-11-10"
REPO = Path("/content/Tesi_OpenRCA_OOD_Audit")  # NON tocca il clone sperimentale precedente
DATA = Path("/content/drive/MyDrive/Tesi_OpenRCA")

drive.mount("/content/drive")

if not (DATA / "data/processed/split_market_v1.csv").is_file():
    raise FileNotFoundError("Non trovo split_market_v1.csv nel Drive previsto")
if not (DATA / "data/processed/benchmark_inputs_market.csv").is_file():
    raise FileNotFoundError("Non trovo benchmark_inputs_market.csv nel Drive previsto")

if REPO.exists():
    if not (REPO / ".git").is_dir():
        raise RuntimeError(f"Directory esistente non Git: {REPO}. Non sovrascrivo.")
    actual_branch = subprocess.check_output(
        ["git", "-C", str(REPO), "branch", "--show-current"], text=True
    ).strip()
    if actual_branch != BRANCH:
        raise RuntimeError(f"Branch inatteso {actual_branch!r}; non faccio checkout/reset automatici")
    local_changes = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True
    ).strip()
    if local_changes:
        raise RuntimeError("Clone con modifiche locali: non eseguo pull automatico")
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(
        ["git", "clone", "--depth", "1", "--single-branch", "--branch", BRANCH,
         REPO_URL, str(REPO)],
        check=True,
    )

print("\n=== FASE 2 — PREFLIGHT OOD (SOLA LETTURA) ===")
print("Branch:", BRANCH)
print("Commit:", subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
).strip())
env = dict(os.environ)
env["PYTHONPATH"] = str(REPO / "src")
subprocess.run(
    [sys.executable, "-m", "evaluation.ood_holdout_preflight",
     "--data-dir", str(DATA)],
    cwd=str(REPO),
    env=env,
    check=True,
)
print("\nEsito: nessuna chiamata Gemini/Claude, nessuno scoring, nessuna modifica ai risultati.")


### Cosa fare con l'output

Condividere il JSON con l'assistente. Solo dopo il controllo dell'inventario e delle eventuali sovrapposizioni si progetterà la partizione per classe e per discovery/test. **Non lanciare Gemini, Sonnet o Haiku da questa demo.**
